In [8]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp

In [5]:
x = np.loadtxt(r'.\Data\Kuramoto\Time_series.csv', delimiter=',')
x_dot = np.loadtxt(r'.\Data\Kuramoto\Time_series_derivative.csv', delimiter=',')

In [6]:
Gamma = 1
r = 2

In [50]:
N = x.shape[1]
K_est = np.zeros((N, N))

# Initialize and normalize the Fourier Coefficients
Lambda_initial = np.ones(2 * r)
Lambda_initial = (1 / np.linalg.norm(Lambda_initial)) * Lambda_initial

Iteration_number = np.zeros((N, Gamma))
Lambda_hat = Lambda_initial
Lambda_each_node = np.zeros((N * Gamma, 2 * r))
def est_F(x):
    return np.ones((x.shape[0], 1))

def est_Z(x, r, node_index):
    """ 
    this function is used to estimate the Z matrix in the optimization problem
    
    x: (M, N): the time series data
    r: the number of Fourier Coefficients
    node_index: the index of the node that we want to estimate the Z matrix
    
    return:  
    Z: (M * (N - 1), r ^ 2): coupling basis
    
    formula: $$Z_i = [sin(k * (x_i - x_j)), cos(k * (x_i - x_j))]_{j=1}^{N}$$ where i != j
    then we have Z = [Z_1, Z_2, ..., Z_N] so Z.shape = (M * (N - 1), r ^ 2)
    """
    M, N = x.shape
    Z = np.zeros((M * (N - 1), 2 * r))
    
    x_temp = np.delete(x, node_index, axis=1)
    for t in range(M):
        Z_i = np.zeros((N - 1, 2 * r))
        
        for k in range(r):
            Diff = x[t, node_index] - x_temp[t, :]
            Z_i[:, 2 * k:2 * k + 2] = np.column_stack((np.sin(k * Diff), np.cos(k * Diff)))
        
        Z[t * (N - 1):(t + 1) * (N - 1), :] = Z_i
    
    return Z

def test_est_Z():
    M = 5
    N = 3
    r = 2
    node_index = 1
    
    x = np.random.rand(M, N)
    
    Z = est_Z(x, r, node_index)
    print("M:", M)
    print("N:", N)
    print("r:", r)
    print("Z:", Z.shape)
    assert Z.shape == (M * (N - 1), 2*r)

test_est_Z()

def estimate_topology(x_dot, F, Z, Lambda, N):
    """ 
    x_dot(M, N): the derivative of the time series data, M is the number of time points
    F(r, N): the Fourier basis
    Z(N - 1, r ^ 2): the coupling basis
    Lambda(2 * r, 1): the Fourier Coefficients
    N: the number of nodes
    """
    M = x_dot.shape[0]
    X = np.zeros((M, N-1))
    
    for i in range(M):
        X[i, :] = (Z[(i)*(N-1):(i+1)*(N-1), :] @ Lambda).T
    
    Temp = np.eye(M) - (F @ np.linalg.pinv(F))
    
    # Define the optimization variables, this problem is a non-negative least square problem
    B_hat = cp.Variable((X.shape[1], x_dot.shape[1]), nonneg=True)
    constraints = [B_hat >= 0]
    # Define the objective function: ||X @ B_hat - x_dot||_2
    objective = cp.Minimize(cp.norm(Temp @ X @ B_hat - Temp @ x_dot, 'fro'))
    problem = cp.Problem(objective, constraints)
    
    # Solve the problem
    problem.solve()
    
    B_hat_value = B_hat.value
    A_hat = np.linalg.pinv(F) @ (x_dot - X @ B_hat_value)
    
    return B_hat_value, A_hat

M: 5
N: 3
r: 2
Z: (10, 4)


In [54]:

def test_estimate_topology():
    M = 5
    N = 3
    r = 2
    Lambda = np.random.rand(2 * r, 1)
    F = est_F(x_dot)
    Z = est_Z(x_dot, r, 0)
    B_hat, A_hat = estimate_topology(x_dot, F, Z, Lambda, N)
    print("B_hat:", B_hat.shape)
    print("A_hat:", A_hat.shape)
    
test_estimate_topology()

B_hat: (2, 100)
A_hat: (1, 100)
